<a href="https://colab.research.google.com/github/carlymariec/document_processing_pipeline/blob/main/Document_Processing_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Environment Setup & Dependency Installation
First, we need to install all the system-level and Python libraries required for high-precision OCR, PDF manipulation, and metadata extraction.

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr tesseract-ocr-eng libtesseract-dev libleptonica-dev pkg-config ghostscript python3-tk poppler-utils
!pip install pytesseract pdf2image PyMuPDF pikepdf pdfplumber Pillow exifread

## 2. Imports and Initial Configuration
Let's import all the necessary modules for our forensic suite.

In [ ]:
import os
import hashlib
import pytesseract
from pdf2image import convert_from_path
import fitz  # PyMuPDF
import pikepdf
import pdfplumber
from PIL import Image
import exifread
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import io

## 3. Utility Functions: Hashing & Metadata Extraction
These functions handle extracting embedded data and calculating cryptographic hashes for authenticity.

In [ ]:
def calculate_hash(file_path, algorithm='sha256'):
    """Calculates the cryptographic hash of a file."""
    hash_func = hashlib.new(algorithm)
    with open(file_path, 'rb') as f:
        while chunk := f.read(8192):
            hash_func.update(chunk)
    return hash_func.hexdigest()

def extract_metadata(file_path):
    """Extracts deep metadata, EXIF, and Hex signatures from PDF or Image files."""
    metadata = {}
    ext = os.path.splitext(file_path)[1].lower()

    try:
        # Extract Hex Signature (first 16 bytes)
        with open(file_path, 'rb') as f:
            header = f.read(16)
            metadata['Hex_Signature'] = header.hex(' ').upper()

        if ext == '.pdf':
            doc = fitz.open(file_path)
            metadata.update(doc.metadata)
            # Attempt to extract XML metadata (XMP)
            try:
                 xmp = doc.get_xml_metadata()
                 if xmp: metadata['XMP'] = xmp
            except: pass
            doc.close()
        elif ext in ['.tiff', '.tif', '.jpg', '.jpeg', '.png']:
             with open(file_path, 'rb') as f:
                tags = exifread.process_file(f)
                for tag in tags.keys():
                    if tag not in ('JPEGThumbnail', 'TIFFThumbnail', 'Filename', 'EXIF MakerNote'):
                        metadata[f"EXIF_{tag}"] = str(tags[tag])
    except Exception as e:
        metadata['Extraction Error'] = str(e)

    return metadata

def save_metadata(file_path, metadata, file_hash):
    """Saves metadata and hash to a text file."""
    base_name = os.path.splitext(file_path)[0]
    meta_file = f"{base_name}_metadata.txt"
    with open(meta_file, 'w') as f:
        f.write(f"File: {os.path.basename(file_path)}\n")
        f.write(f"SHA-256 Hash: {file_hash}\n")
        f.write("-"*40 + "\n")
        f.write("Deep Metadata, EXIF & Hex Data:\n")
        for k, v in metadata.items():
            f.write(f"{k}: {v}\n")
    return meta_file

## 4. File Upload Interface
This creates the button to upload multiple files (PDFs, TIFFs, etc.) into the Colab environment.

In [ ]:
uploaded_files_registry = []

def handle_upload(change):
    clear_output()
    print("Uploading files...")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        # Save file to local storage
        with open(filename, 'wb') as f:
            f.write(content)

        # Initial integrity check and logging
        file_hash = calculate_hash(filename)
        meta = extract_metadata(filename)
        meta_file = save_metadata(filename, meta, file_hash)

        uploaded_files_registry.append({
            'original_name': filename,
            'hash': file_hash,
            'metadata_file': meta_file,
            'status': 'Uploaded & Hashed'
        })
        print(f"Processed: {filename} | Hash: {file_hash[:10]}... | Metadata saved to: {meta_file}")

    print("\nUpload phase complete. Ready for next steps.")

upload_btn = widgets.Button(
    description='Upload Documents (PDF/TIFF/IMG)',
    button_style='info',
    tooltip='Click to upload multiple files',
    layout=widgets.Layout(width='300px')
)
upload_btn.on_click(handle_upload)

display(upload_btn)

## 4.5 Archive Extraction
Scans uploaded files for `.zip` archives, extracts their contents, and processes valid documents (TIFFs, PDFs, Images) into the registry for the next pipeline stages.

In [ ]:
import zipfile

def process_zip_archives(registry):
    print("Scanning for ZIP archives to extract...")
    new_registry = []

    for item in registry:
        file_path = item['original_name']
        ext = os.path.splitext(file_path)[1].lower()

        if ext == '.zip':
            print(f"Extracting {file_path}...")
            extract_dir = f"{os.path.splitext(file_path)[0]}_extracted"
            os.makedirs(extract_dir, exist_ok=True)

            try:
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(extract_dir)

                # Process extracted files
                for root, _, files in os.walk(extract_dir):
                    for file in files:
                        # Ignore hidden/system files
                        if file.startswith('.') or file.startswith('__'):
                            continue

                        extracted_path = os.path.join(root, file)
                        ext_lower = os.path.splitext(file)[1].lower()

                        # Only add relevant documents
                        if ext_lower in ['.tiff', '.tif', '.pdf', '.jpg', '.jpeg', '.png']:
                            file_hash = calculate_hash(extracted_path)
                            meta = extract_metadata(extracted_path)
                            meta_file = save_metadata(extracted_path, meta, file_hash)

                            new_registry.append({
                                'original_name': extracted_path,
                                'hash': file_hash,
                                'metadata_file': meta_file,
                                'status': 'Extracted & Hashed'
                            })
                            print(f"  - Extracted & Processed: {file} | Hash: {file_hash[:10]}...")
                item['status'] = 'ZIP Extracted'
                print(f"Finished extracting {file_path}.")
            except Exception as e:
                print(f"Error extracting {file_path}: {e}")
        else:
            # Not a zip, keep it in the registry
            new_registry.append(item)

    print("Archive extraction phase complete.")
    return new_registry

# Run ZIP extraction
uploaded_files_registry = process_zip_archives(uploaded_files_registry)

## 5. Format Normalization
Converts any uploaded image/TIFF files into standardized PDFs so the rest of the pipeline handles a uniform file format.

In [ ]:
def normalize_to_pdf(registry):
    print("Starting Format Normalization...")
    for item in registry:
        file_path = item['original_name']
        ext = os.path.splitext(file_path)[1].lower()

        if ext == '.pdf':
            print(f"Skipping {file_path}, already PDF.")
            item['normalized_pdf'] = file_path
            continue

        if ext in ['.tiff', '.tif', '.jpg', '.jpeg', '.png']:
            print(f"Converting {file_path} to PDF...")
            pdf_path = f"{os.path.splitext(file_path)[0]}_normalized.pdf"

            try:
                # Use PyMuPDF to convert image to PDF
                doc = fitz.open()
                img_doc = fitz.open(file_path)
                pdfbytes = img_doc.convert_to_pdf()
                img_pdf = fitz.open("pdf", pdfbytes)
                doc.insert_pdf(img_pdf)
                doc.save(pdf_path)
                doc.close()
                img_doc.close()

                item['normalized_pdf'] = pdf_path
                item['status'] = 'Normalized to PDF'
                print(f"Successfully converted to {pdf_path}")
            except Exception as e:
                print(f"Error converting {file_path}: {e}")

    print("Normalization complete.")
    return registry

# Run normalization (you would call this after uploads are finished)
uploaded_files_registry = normalize_to_pdf(uploaded_files_registry)

## 6. High-Precision OCR Extraction
Converts PDF pages into 300 DPI high-resolution images and uses Tesseract to extract text with high precision.

In [ ]:
def perform_ocr(registry):
    print("Starting High-Precision OCR Extraction...")
    for item in registry:
        if 'normalized_pdf' not in item:
            continue

        pdf_path = item['normalized_pdf']
        print(f"Running OCR on {pdf_path}...")

        extracted_pages = []
        try:
            # Convert PDF to high-res images (300 DPI for precision)
            images = convert_from_path(pdf_path, dpi=300)
            for i, img in enumerate(images):
                # Use tesseract to extract text
                text = pytesseract.image_to_string(img)
                extracted_pages.append({'page': i+1, 'text': text})
                print(f"  - Extracted text from page {i+1}")

            item['ocr_text'] = extracted_pages
            item['status'] = 'OCR Complete'
            print(f"Finished OCR for {pdf_path}.")
        except Exception as e:
            print(f"OCR Error on {pdf_path}: {e}")

    print("OCR phase complete.")
    return registry

# Run OCR
uploaded_files_registry = perform_ocr(uploaded_files_registry)

## 7. Human-in-the-Loop Verification
An interactive interface to review and correct the extracted OCR text before embedding it as a hidden, searchable layer.

In [ ]:
def verification_ui(registry):
    if not registry:
        print("No files to verify. Please run upload and OCR first.")
        return

    # We'll just review the first document for this demo structure
    doc = registry[0]
    if 'ocr_text' not in doc:
        print(f"No OCR text found for {doc['original_name']}")
        return

    print(f"Verifying document: {doc['original_name']}")

    # Setup UI elements
    page_selector = widgets.Dropdown(
        options=[(f"Page {p['page']}", i) for i, p in enumerate(doc['ocr_text'])],
        description='Page:',
    )

    text_area = widgets.Textarea(
        value=doc['ocr_text'][0]['text'],
        placeholder='Extracted text will appear here...',
        description='OCR Text:',
        layout=widgets.Layout(width='100%', height='300px')
    )

    save_btn = widgets.Button(description="Save & Approve Page", button_style='success')

    def on_page_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            idx = change['new']
            text_area.value = doc['ocr_text'][idx]['text']

    def on_save_click(b):
        idx = page_selector.value
        doc['ocr_text'][idx]['text'] = text_area.value
        doc['status'] = 'Verified'
        print(f"Saved approved text for Page {doc['ocr_text'][idx]['page']}.")

    page_selector.observe(on_page_change)
    save_btn.on_click(on_save_click)

    display(widgets.VBox([page_selector, text_area, save_btn]))

# Run verification UI
verification_ui(uploaded_files_registry)

## 8. Embedding Searchable Hidden Text Layer
Injects the verified OCR text back into the PDF as an invisible layer so the document can be searched and highlighted.

In [ ]:
def embed_hidden_text(registry):
    print("Starting Hidden Text Embedding...")
    for item in registry:
        if 'ocr_text' not in item:
            print(f"Skipping {item['original_name']} - No OCR text available.")
            continue

        pdf_path = item['normalized_pdf']
        output_pdf = f"{os.path.splitext(item['original_name'])[0]}_searchable.pdf"
        print(f"Embedding text into {output_pdf}...")

        try:
            # Open the normalized PDF with PyMuPDF
            doc = fitz.open(pdf_path)

            for page_data in item['ocr_text']:
                page_num = page_data['page'] - 1
                text = page_data['text']

                if page_num < len(doc):
                    page = doc[page_num]
                    # Insert text as invisible (render mode 3)
                    # We place it spanning the page to ensure searchability
                    rect = page.rect
                    page.insert_textbox(
                        rect,
                        text,
                        fontsize=11,
                        fontname="helv",
                        color=(0,0,0),
                        render_mode=3,  # 3 = Invisible text
                        align=fitz.TEXT_ALIGN_LEFT
                    )

            # Save the new searchable PDF
            doc.save(output_pdf)
            doc.close()

            # Pass through pikepdf to optimize and fix any structural issues
            with pikepdf.Pdf.open(output_pdf, allow_overwriting_input=True) as pdf:
                pdf.save(output_pdf)

            item['final_pdf'] = output_pdf
            item['status'] = 'Finalized'

            # Calculate final hash
            item['final_hash'] = calculate_hash(output_pdf)
            print(f"Successfully created searchable PDF: {output_pdf}")
            print(f"Final SHA-256 Hash: {item['final_hash']}")

        except Exception as e:
            print(f"Error embedding text for {pdf_path}: {e}")

    return registry

# Run embedding
uploaded_files_registry = embed_hidden_text(uploaded_files_registry)

## 9. Final Pipeline Export
Packages the final PDFs, their metadata logs, and provides a download link for the completed batch.

In [ ]:
import shutil
import os

def export_results(registry):
    print("Preparing final export...")
    export_dir = "Forensic_Processed_Docs"
    os.makedirs(export_dir, exist_ok=True)

    for item in registry:
        if 'final_pdf' in item:
            pdf_basename = os.path.basename(item['final_pdf'])
            meta_basename = os.path.basename(item['metadata_file'])

            # Move final PDF
            shutil.copy(item['final_pdf'], os.path.join(export_dir, pdf_basename))
            # Move Metadata
            shutil.copy(item['metadata_file'], os.path.join(export_dir, meta_basename))

            # Append final hash to metadata log
            with open(os.path.join(export_dir, meta_basename), 'a') as f:
                f.write(f"\nFinal Processed SHA-256 Hash: {item['final_hash']}\n")

    # Create a zip archive
    shutil.make_archive("Forensic_Processed_Docs", 'zip', export_dir)
    print("Export packaged successfully.")

    # Provide download button
    download_btn = widgets.Button(
        description='Download Processed Files (ZIP)',
        button_style='success',
        icon='download',
        layout=widgets.Layout(width='300px')
    )

    def on_download_click(b):
        files.download("Forensic_Processed_Docs.zip")

    download_btn.on_click(on_download_click)
    display(download_btn)

# Run export
export_results(uploaded_files_registry)

def erase_workspace():
    """Erases all uploaded and processed files to prepare for a new batch."""
    print("\n--- Erasing Workspace ---")
    keep_list = ['.config', 'sample_data']
    for item in os.listdir('/content'):
        if item not in keep_list:
            item_path = os.path.join('/content', item)
            try:
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.unlink(item_path)
                elif os.path.isdir(item_path):
                    shutil.rmtree(item_path)
            except Exception as e:
                print(f"Failed to delete {item_path}: {e}")

    # Reset global registry
    global uploaded_files_registry
    uploaded_files_registry = []
    print("All files deleted and registry reset. Ready for new batch!")

# --- UNCOMMENT THE LINE BELOW TO CLEAR FILES FOR THE NEXT BATCH ---
# erase_workspace()